In [1]:
import kaleidocell
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells"
os.makedirs(OUT_DIR, exist_ok=True)

SELECTED_MPS = ["MP1", "MP2", "MP7", "MP8", "MP9", "MP11", "MP14", "MP15", "MP19", "MP20", "MP24"]

checkpoint("Loading data")
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")
checkpoint(f"Data loaded: {adata.n_obs} cells")

checkpoint("Loading mp_scores")
mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea_c7/mp_scores.csv", index_col=0)
checkpoint(f"mp_scores loaded: {mp_scores.shape}")

[2026-08-05 17:02:28] Loading data
[2026-08-05 17:03:11] Data loaded: 633710 cells
[2026-08-05 17:03:11] Loading mp_scores
[2026-08-05 17:03:14] mp_scores loaded: (633710, 30)


In [2]:
checkpoint(f"mp_scores loaded: {mp_scores.shape}")
print("Columns:", list(mp_scores.columns[:15]))
print("Index sample:", list(mp_scores.index[:15]))

[2026-08-05 17:06:32] mp_scores loaded: (633710, 30)
Columns: ['MP1_score', 'MP2_score', 'MP3_score', 'MP4_score', 'MP5_score', 'MP6_score', 'MP7_score', 'MP8_score', 'MP9_score', 'MP10_score', 'MP11_score', 'MP12_score', 'MP13_score', 'MP14_score', 'MP15_score']
Index sample: ['PJ017_2-0', 'PJ017_16-0', 'PJ017_17-0', 'PJ017_34-0', 'PJ017_40-0', 'PJ017_45-0', 'PJ017_46-0', 'PJ017_56-0', 'PJ017_59-0', 'PJ017_66-0', 'PJ017_73-0', 'PJ017_80-0', 'PJ017_83-0', 'PJ017_85-0', 'PJ017_87-0']


In [3]:
print(adata.obs_names[:5].tolist())
missing = set(adata.obs_names) - set(mp_scores.index)
print(f"Missing cells: {len(missing)} / {adata.n_obs}")

['PJ017_2-0', 'PJ017_16-0', 'PJ017_17-0', 'PJ017_34-0', 'PJ017_40-0']
Missing cells: 0 / 633710


In [1]:
import kaleidocell
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

os.environ['TZ'] = 'Europe/Berlin'
import time
time.tzset()

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells"
os.makedirs(OUT_DIR, exist_ok=True)

SELECTED_MPS = ["MP1", "MP2", "MP7", "MP8", "MP9", "MP11", "MP14", "MP15", "MP19", "MP20"]

checkpoint("Loading data")
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_cells_extended_gbmap_hgnc.h5ad")
checkpoint(f"Data loaded: {adata.n_obs} cells")

checkpoint("Loading mp_scores")
mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea_c7/mp_scores.csv", index_col=0)
checkpoint(f"mp_scores loaded: {mp_scores.shape}")

missing_cells = set(adata.obs_names) - set(mp_scores.index)
if missing_cells:
    raise ValueError(f"{len(missing_cells)} cells in adata not found in mp_scores index")

mp_scores = mp_scores.loc[adata.obs_names]
all_score_cols = [c for c in mp_scores.columns if c.endswith("_score")]
mp_scores_renamed = mp_scores[all_score_cols].copy()
mp_scores_renamed.columns = [c.replace("_score", "") for c in all_score_cols]

checkpoint("Computing dominant MP across ALL MPs")
dominant_MP_all = mp_scores_renamed.idxmax(axis=1)
max_scores_all = mp_scores_renamed.max(axis=1)

threshold = np.percentile(max_scores_all, 75)
checkpoint(f"Using adaptive threshold (75th percentile): {threshold:.4f}")

dominant_MP_all = np.where(max_scores_all.values > threshold, dominant_MP_all.values, "Other")


adata.obs["dominant_MP_all"] = dominant_MP_all

checkpoint(f"Distribution:\n{adata.obs['dominant_MP_all'].value_counts()}")

[2026-08-06 16:44:04] Loading data
[2026-08-06 16:44:39] Data loaded: 633710 cells
[2026-08-06 16:44:39] Loading mp_scores
[2026-08-06 16:44:41] mp_scores loaded: (633710, 30)
[2026-08-06 16:44:42] Computing dominant MP across ALL MPs
[2026-08-06 16:44:42] Using adaptive threshold (75th percentile): 2.6886
[2026-08-06 16:44:42] Distribution:
dominant_MP_all
Other    475282
MP12     104285
MP13      36798
MP10      10964
MP3        4463
MP21       1668
MP30        100
MP28         92
MP5          55
MP4           2
MP16          1
Name: count, dtype: int64


In [2]:
print(adata.obs["dominant_MP_all"].unique()[:10])

['Other' 'MP13' 'MP3' 'MP10' 'MP21' 'MP28' 'MP5' 'MP12' 'MP30' 'MP4']


In [3]:
print(adata.obs["dominant_MP_all"].value_counts())

dominant_MP_all
Other    475282
MP12     104285
MP13      36798
MP10      10964
MP3        4463
MP21       1668
MP30        100
MP28         92
MP5          55
MP4           2
MP16          1
Name: count, dtype: int64
